# 1.4 Computational graphs

`pytorch` is an engine for autodifferentiation of large matrix structures (tensors). The engine as a whole is referred to as `autograd`.

We can compute the derivative of any function evaluated at some point in the input space.

For instance, we may have

$$
\begin{align}
f(x,y) &= log(x+2y)^3\\
\frac{\partial f}{\partial x} &= \frac{3 \log(x+2y)^2}{x+2y}\\
\frac{\partial f}{\partial y} &=\frac{6 \log(x+2y)^2}{x+2y}
\end{align}
$$

which, if we evaluate at point $x=2$ and $y=3$ we get

$$
\begin{align}
f(x,y) &= 8.992\\
\frac{\partial f}{\partial x} &= 1.622\\
\frac{\partial f}{\partial y} &= 3.243
\end{align}
$$

In `pytorch` this is done as follows. The key is the `.backward` function which computes all expressions


In [511]:
import torch

# Create tensors
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

f = torch.log(x+2*y)**3
f.backward()

print ('f(x,y)      =', f)
print ('df/dx (x,y) =', x.grad)
print ('df/dy (x,y) =', y.grad)

f(x,y)      = tensor(8.9917, grad_fn=<PowBackward0>)
df/dx (x,y) = tensor(1.6215)
df/dy (x,y) = tensor(3.2431)


as opposed to symbolic packages (also called _computer algebra_) `pytorch` does not compute derivative expressions explicitly, which is very computationally and memory expensive.

For instance, observe the difference in compute times between `sympy` and `pytorch`

In [521]:
import sympy as sy

x, y = sy.symbols("x y")
f = sy.log(x+2*y)**3

f.diff(x)

3*log(x + 2*y)**2/(x + 2*y)

In [522]:
# the value of the function
print ("f(x,y)       ", f.subs({x: 2., y: 3.}))
print ("d_f/d_x      ", f.diff(x).subs({x: 2., y: 3.}))
print ("d_f/d_y      ", f.diff(y).subs({x: 2., y: 3.}))


f(x,y)        8.99166560370109
d_f/d_x       1.62152892197393
d_f/d_y       3.24305784394786


In [523]:
%%timeit
## with sympy
x, y = sy.symbols("x y")
f = sy.log(x+2*y)**3
df_dx = f.diff(x).subs({x: 2., y: 3.})

168 μs ± 3.7 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [524]:
%%timeit

## with pytorch
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

f = torch.log(x+2*y)**3
f.backward()
df_dx = x.grad


66.1 μs ± 1.65 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


## Differentiation in `pytorch`

`pytorch` backpropagates the gradients, using the computational graph of the expression we want to derivate. There are two main steps:

- the **forward pass**: which is the natural flow of our code to compute the output value
- the **backward pass**: which, when invoked, the gradients are computed

`pytorch` builds the computational graph as the forward pass is progressing, as the lines of your code are being executed. 

The backward pass has to be explicitely invoked when we are finished building the expression we want.

We use the same example expression as above

In [563]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

f = torch.log(x+2*y)**3
f

tensor(8.9917, grad_fn=<PowBackward0>)

`pytorch` stores tha graph as it is computing it in the `graph_fn` function on each tensor along the way. And then it can be accessed starting off from the end of the graph (the output at $f$)

In [529]:
def print_graph(t, indent=2):
    print (" "*indent, t.__class__.__name__)
    if 'next_functions' in dir(t) and t.next_functions is not None:
        for ti in t.next_functions:
            p(ti[0], indent+2)

In [530]:
print_graph(f.grad_fn)

   PowBackward0
     LogBackward0
       AddBackward0
         AccumulateGrad
         MulBackward0
           AccumulateGrad
           NoneType


however it is not until we call `.backward` that it computes the gradients

In [535]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

f = torch.log(x+2*y)**3

## before calling backward
print ('gradients before calling backward', x.grad, y.grad)
f.backward()
print ('gradients after calling backward ', x.grad, y.grad)


gradients before calling backward None None
gradients after calling backward  tensor(1.6215) tensor(3.2431)


observe that with `requires_grad` we can tell `pytorch` for what input variables we want to have the gradient computed. This will come handy when we don't need all gradients (such as when freezing part of a neural network when training)

In [537]:
x = torch.tensor(2.0, requires_grad=False)
y = torch.tensor(3.0, requires_grad=True)
f = torch.log(x+2*y)**3

f.backward()
x.grad, y.grad


(None, tensor(3.2431))

## Reverse differentiation

What `pytorch` actually does is called **reverse differentiation**, computing the gradients traversing backwards the graph built in the forward pass.

Reverse differentiation is specially efficient when the dimensionality of the input is much larger than the dimensionality of the output. Whis is precisely the case in Deep Learning.

See https://arxiv.org/abs/1502.05767, for a detailed example and the justification for this approach.


We can name the intermediate steps in our function to understand what is happening. Given $f(x,y) = log(x+2y)^3$, we can name and evaluate each intermediate step like this

$$
\begin{aligned}[rrr]
v_1 &= x+2y &&= 8\\
v_2 &= log(v_1) &&= 2.0794\\
v_3 &= v_2 \;^ 3 &&= 8.9917\\
f &=v_3 &&= 8.9917
\end{aligned}
$$

This is the **forward pass**, simply the calculation of the output value.

And remember the **chain rule**


$$
\frac{\partial f}{\partial x} = \frac{\partial f}{\partial v_3}\frac{\partial v_3}{\partial v_2}\frac{\partial v_2}{\partial v_1}\frac{\partial v_1}{\partial x}
$$

which we can also breakdown in steps starting from the output


$$
\begin{aligned}
step\;1 &&\frac{\partial f}{\partial v_3} && && = 1 &&\\\\
step\;2 &&\frac{\partial f}{\partial v_2} &&= \frac{\partial f}{\partial v_3}\frac{\partial v_3}{\partial v_2}&&=12.9722\\\\
step\;3 &&\frac{\partial f}{\partial v_1} &&= \frac{\partial f}{\partial v_2}\frac{\partial v_2}{\partial v_1}&&=1.6215\\\\
step\;4 &&\frac{\partial f}{\partial x} &&= \frac{\partial f}{\partial v_1}\frac{\partial v_1}{\partial x}&&=1.6215 \\\\
        &&\frac{\partial f}{\partial y} &&= \frac{\partial f}{\partial v_1}\frac{\partial v_1}{\partial y}&&=3.2431
\end{aligned}
$$


and using the values of $v_i$ above, we have that
$$
\frac{\partial v_3}{\partial v_2} = 3v_2\;^2 = 12.9722 \;\;\;\;; \;\;\;\; 
\frac{\partial v_2}{\partial v_1} = \frac{1}{v_1} = \frac{1}{8} \;\;\;\;; \;\;\;\;
\frac{\partial v_1}{\partial x} = 1 \;\;\;\; ; \;\;\;\;
\frac{\partial v_1}{\partial y} = 2
$$

which is what we get with `pytorch`

In [757]:
# Create tensors
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

v1 = x+2*y
v2 = torch.log(v1)
v3 = v2**3
f = v3

print ('forward pass (the value of the function):', f.detach().numpy())


for vi in [v1,v2,v3]:
    vi.retain_grad()

f.backward()
print ('df/dv3   ', v3.grad)
print ('df/dv2   ', v2.grad)
print ('df/dv1   ', v1.grad)
print ('df/dx    ', x.grad)
print ('df/dy    ', y.grad)

forward pass (the value of the function): 8.991666
df/dv3    tensor(1.)
df/dv2    tensor(12.9722)
df/dv1    tensor(1.6215)
df/dx     tensor(1.6215)
df/dy     tensor(3.2431)


## `pytorch` uses Dynamic Computational Graphs

If we re doing a training loop, the copmutational graph is build at eery iteration of the loop. This might seem unefficient (as oposed to TensorFlow), but the flexibility pays off. This was not so evident in the early days of DL, when TF and torch were being developed (circa 2017).

## Deep Learning is about matrices

We are interested in the gradients at the weights of a perceptron. For instance, we have

- a batch with 2 data elements, each one with three features
- a perceptron with one layer with 5 neurons and a single output neuron

In [760]:
# an input
X = torch.randn(2,3)
y = torch.randn(2,1)

# weights for hidden layer
W1 = torch.randn(5, 3, requires_grad=True)
b1 = torch.randn(5, requires_grad=True)

# weights for the output layer
W2 = torch.randn(1, 5, requires_grad=True)
b2 = torch.randn(1, requires_grad=True)

# check weight sizes
[i.shape for i in (X,W1,W2)]

[torch.Size([2, 3]), torch.Size([5, 3]), torch.Size([1, 5])]

In [761]:
# set grads to None (in case of repeated execution, since
# .backward accumulates the gradiens)

for w in (W1,W2,b1,b2):
    w.grad = None

# the forward pass (with loss)
output = torch.sigmoid(torch.sigmoid(X.matmul(W1.T)+b1).matmul(W2.T)+b2)
loss = ((output - y)**2).mean()

output, loss

(tensor([[0.6236],
         [0.4012]], grad_fn=<SigmoidBackward0>),
 tensor(0.6558, grad_fn=<MeanBackward0>))

In [762]:
# the backward pass
loss.backward()

In [763]:
# and we have the gradients at each weight
W1.grad, b1.grad, W2.grad, b2.grad

(tensor([[1.8084e-05, 1.5628e-03, 3.1578e-03],
         [1.7167e-03, 1.2207e-02, 2.1815e-02],
         [4.8709e-04, 8.4852e-03, 1.6442e-02],
         [1.0299e-02, 2.1911e-02, 2.6098e-02],
         [3.3375e-03, 4.8271e-02, 9.2508e-02]]),
 tensor([-0.0026, -0.0198, -0.0141, -0.0321, -0.0800]),
 tensor([[0.2073, 0.1111, 0.0162, 0.0689, 0.1337]]),
 tensor([0.3083]))

which is nicely packaged under the `torch.nn` modules

In [764]:
from torch import nn

In [765]:
class Perceptron(nn.Module):
    def __init__(self, input_size):
        super(Perceptron, self).__init__()
        # nn.Linear implements the weighted sum and bias
        self.fc1 = nn.Linear(input_size, 5) 
        self.fc2 = nn.Linear(5, 1)

    def forward(self, x):
        # Apply a sigmoid activation for binary output
        out = torch.sigmoid(self.fc2(torch.sigmoid(self.fc1(x)))) 
        return out

In [869]:
m = Perceptron(input_size=3)

In [870]:
# observe we have the same parameters shapes as above
[i.shape for i in m.parameters()]

[torch.Size([5, 3]), torch.Size([5]), torch.Size([1, 5]), torch.Size([1])]

In [871]:
# we copy the values from the manual example to get the same result
with torch.no_grad():
    for w,v in zip(m.parameters(), [W1,b1,W2,b2]):
        w.copy_(v)

In [872]:
# same output as above
m(X)

tensor([[0.6236],
        [0.4012]], grad_fn=<SigmoidBackward0>)

In [873]:
# the forward pass
loss = ((m(X) - y)**2).mean()
loss

tensor(0.6558, grad_fn=<MeanBackward0>)

In [874]:
# the backward pass
loss.backward()

In [875]:
# and we get the same gradients
[w.grad for w in m.parameters()]

[tensor([[1.8084e-05, 1.5628e-03, 3.1578e-03],
         [1.7167e-03, 1.2207e-02, 2.1815e-02],
         [4.8709e-04, 8.4852e-03, 1.6442e-02],
         [1.0299e-02, 2.1911e-02, 2.6098e-02],
         [3.3375e-03, 4.8271e-02, 9.2508e-02]]),
 tensor([-0.0026, -0.0198, -0.0141, -0.0321, -0.0800]),
 tensor([[0.2073, 0.1111, 0.0162, 0.0689, 0.1337]]),
 tensor([0.3083])]

## Optimizers

optimizers traverse the computational graph after the backward pass (when all the gradients have been stored) and update the weights according to their gradients.

In [876]:
from torch import optim

In [899]:
m = Perceptron(input_size=3)
learning_rate = 0.5

# simplest optimizer, just  substracting the gradient scaled by a learning rate
optimizer = optim.SGD(m.parameters(), lr=learning_rate)

In [900]:
loss = ((m(X) - y)**2).mean()
optimizer.zero_grad()
loss.backward()

In [901]:
# manual computation of the new weights
new_weights_manual = [w - learning_rate * w.grad for w in m.parameters()]

In [902]:
# allow the optimizer to compute and the new weights
optimizer.step()
list(m.parameters())

[Parameter containing:
 tensor([[ 0.1469, -0.0584,  0.4112],
         [-0.4362, -0.4351,  0.3527],
         [-0.3609,  0.0819, -0.3203],
         [ 0.3338, -0.5543,  0.2777],
         [ 0.5552,  0.3934,  0.0814]], requires_grad=True),
 Parameter containing:
 tensor([-0.2756,  0.5537, -0.2676,  0.5315, -0.3498], requires_grad=True),
 Parameter containing:
 tensor([[-0.1658,  0.1770, -0.4272, -0.4978, -0.3666]], requires_grad=True),
 Parameter containing:
 tensor([-0.2356], requires_grad=True)]

In [903]:
# which correspond to those computed manually
new_weights_manual

[tensor([[ 0.1469, -0.0584,  0.4112],
         [-0.4362, -0.4351,  0.3527],
         [-0.3609,  0.0819, -0.3203],
         [ 0.3338, -0.5543,  0.2777],
         [ 0.5552,  0.3934,  0.0814]], grad_fn=<SubBackward0>),
 tensor([-0.2756,  0.5537, -0.2676,  0.5315, -0.3498], grad_fn=<SubBackward0>),
 tensor([[-0.1658,  0.1770, -0.4272, -0.4978, -0.3666]], grad_fn=<SubBackward0>),
 tensor([-0.2356], grad_fn=<SubBackward0>)]